# 🥉 Bronze Layer: Real-Time Data Ingestion
This notebook acts as the **Producer** in our streaming pipeline. It fetches real-time market data for **VOO, VYM, and VGT** using the `yfinance` API and lands it as raw JSON files in the Lakehouse landing zone. 

By landing files at a specific frequency, we simulate a continuous data stream for the downstream **Spark Structured Streaming** engine.

### 🛠️ Process Overview
1. **Library Installation:** Installs `yfinance` to interact with market APIs.
2. **Directory Management:** Ensures the `/Files/landing/` path exists in the Lakehouse.
3. **Data Fetching:** Pulls 1-minute interval data for a diversified ETF set (S&P 500, Dividend, and Tech).
4. **Partitioning & Landing:** Saves each ticker as a unique JSON file with a high-resolution timestamp to trigger the Silver layer stream.

In [7]:
%pip install yfinance -q

StatementMeta(, e1640ea8-0362-4a9e-8bb2-ae0afc9573b5, 28, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [8]:
import yfinance as yf
import time
from datetime import datetime
import os

# Define the tickers
tickers = ["VOO", "VYM", "VGT"]
# Define the landing path in your Lakehouse
landing_path = "/lakehouse/default/Files/landing/"

# Create the directory if it doesn't exist
if not os.path.exists(landing_path):
    os.makedirs(landing_path)

def fetch_and_save_stock_data():
    # Fetch real-time data
    data = yf.download(tickers, period="1d", interval="1m", group_by='ticker')
    
    for ticker in tickers:
        # Get the latest price row
        latest_data = data[ticker].tail(1).reset_index()
        latest_data['ticker'] = ticker
        latest_data['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Convert to JSON and save as a unique file to trigger the stream
        file_name = f"{ticker}_{datetime.now().strftime('%Y%m%d%H%M%S')}.json"
        full_path = os.path.join(landing_path, file_name)
        
        latest_data.to_json(full_path, orient="records")
        print(f"Saved {ticker} data to {full_path}")

# Simulate a live feed by running this a few times
for i in range(5):
    fetch_and_save_stock_data()
    print("Waiting for next poll...")
    time.sleep(15) # Wait 15 seconds between polls

StatementMeta(, e1640ea8-0362-4a9e-8bb2-ae0afc9573b5, 30, Finished, Available, Finished, False)

Saved VOO data to /lakehouse/default/Files/landing/VOO_20260225232813.json
Saved VYM data to /lakehouse/default/Files/landing/VYM_20260225232813.json
Saved VGT data to /lakehouse/default/Files/landing/VGT_20260225232813.json
Waiting for next poll...
Saved VOO data to /lakehouse/default/Files/landing/VOO_20260225232828.json
Saved VYM data to /lakehouse/default/Files/landing/VYM_20260225232828.json
Saved VGT data to /lakehouse/default/Files/landing/VGT_20260225232829.json
Waiting for next poll...
Saved VOO data to /lakehouse/default/Files/landing/VOO_20260225232844.json
Saved VYM data to /lakehouse/default/Files/landing/VYM_20260225232844.json
Saved VGT data to /lakehouse/default/Files/landing/VGT_20260225232844.json
Waiting for next poll...
Saved VOO data to /lakehouse/default/Files/landing/VOO_20260225232859.json
Saved VYM data to /lakehouse/default/Files/landing/VYM_20260225232900.json
Saved VGT data to /lakehouse/default/Files/landing/VGT_20260225232900.json
Waiting for next poll...
